# Reaction-Diffusion Simulator

This notebook runs the `ReactionDiffusion` simulator — a coupled PDE system modelling two interacting species $u$ and $v$ with diffusion and nonlinear reaction kinetics.

- **State variables**: `u` and `v` (species concentrations).
- **Conditioning variables**: `beta` (reaction rate strength), `d` (ratio of diffusion coefficients).
- **Dynamics**: spectral (FFT-based) spatial differentiation combined with `scipy` RK45 time integration.
- **Boundary conditions**: periodic in both spatial directions.

### Why this is useful

Reaction-diffusion systems produce a rich variety of spatiotemporal patterns — stripes, spots, spirals, and labyrinthine textures — depending on the parameter regime. This makes them a natural benchmark for learning PDEs that exhibit diverse structure from low-dimensional parameters.


## Governing equations

The two-species reaction-diffusion system takes the form:

$$
\frac{\partial u}{\partial t} = D_u \nabla^2 u + R_u(u, v; \beta),
\qquad
\frac{\partial v}{\partial t} = D_v \nabla^2 v + R_v(u, v; \beta),
$$

where $D_v / D_u = d$ and $R_u$, $R_v$ are the nonlinear reaction terms parameterised by $\beta$.

### Boundary conditions

Periodic in both spatial directions on the domain $[-L/2, L/2]^2$:

$$
u(-L/2, y, t) = u(L/2, y, t), \qquad u(x, -L/2, t) = u(x, L/2, t),
$$

and likewise for $v$.

### Parameters

- **`beta`**: controls the strength of the reaction nonlinearity. Larger values push the system further from equilibrium.
- **`d`**: ratio of diffusion coefficients $D_v / D_u$. Small `d` means species $v$ diffuses much more slowly, favouring pattern formation.


In [ ]:
from IPython.display import HTML

from autosim.experimental.simulations import ReactionDiffusion
from autosim.utils import plot_spatiotemporal_video

In [ ]:
sim = ReactionDiffusion(
    return_timeseries=True,
    log_level="progress_bar",
    n=64,
    L=20,
    T=32.2,
    dt=0.1,
    parameters_range={
        "beta": (1.0, 2.0),
        "d": (0.05, 0.3),
    },
)

batch = sim.forward_samples_spatiotemporal(n=2, random_seed=42)

print("data shape:", batch["data"].shape, "[batch, time, x, y, channels={u, v}]")
print("constant_scalars shape:", batch["constant_scalars"].shape)
print("sampled params (beta, d):", batch["constant_scalars"])

In [ ]:
anim = plot_spatiotemporal_video(
    batch["data"],
    batch_idx=0,
    channel_names=["u", "v"],
    preserve_aspect=True,
)
HTML(anim.to_jshtml())